# UMUD: supported-depth inference with failed-head exclusion

The new **309-row submission candidate has been generated and independently validated**. Both stages ran on Kaggle and produced the same primary CSV verified locally. The pipeline builds on public **prvsiyan V12** (verified public score **0.30108**, versus our existing **0.69948**). Models were trained by the public-source authors; no models were trained here.

**Stage one:** evaluate the weighted PA-versus-depth fit at `clip(0.9, observed minimum depth, observed maximum depth)`, consistently for individual images and similar-frame groups.

**Stage two:** exclude heads whose recorded measurement flag is `ok=False`, then renormalize the original 0.5/0.1 weights over valid heads; stop if every head fails. Seven rows change and **302 remain exactly unchanged**. Five affected images retain only one valid head, which provides no ensemble corroboration. This repairs use of fallback values from failed heads without fitting to labels or leaderboard outcomes.

**Status:** unsubmitted and unscored. Integrity checks do not establish an accuracy gain. The analogous auriclee replay reports 0.29922 but fails its own frozen offline generalization gate.

## Primary submission file

`/kaggle/working/umud_supported_depth_v1/failure_aware/submission.csv`

The original supported-depth output and unbounded control are retained for comparison. Use the failure_aware CSV for this new candidate.

## Inputs and execution

Use Kaggle **GPU T4 ×2 or P100**, enable Internet for the pinned package install, and attach:

1. [UMUD competition](https://www.kaggle.com/competitions/umud-challenge-muscle-architecture-in-ultrasound-data)
2. [watehell original model pair](https://www.kaggle.com/datasets/watehell/umud-r34-unet-weights)
3. [prvsiyan fivefold weights](https://www.kaggle.com/datasets/prvsiyan/umud-r34-5fold-unet-weights)

Use the final executed notebook and run all cells. The final primary output on Kaggle is `/kaggle/working/umud_supported_depth_v1/failure_aware/submission.csv`; the stage-one supported-depth CSV, control, source modules, hashes, calibration and per-head diagnostics are retained separately. The notebook does not submit automatically. `live_run_compatibility/` is a supporting first-stage source snapshot, not a replacement for the executed notebook. Hardening revision 1.1 and its builder are packaged under `extras/future_not_executed/`; they did not produce this CSV.

## Assumptions and limits

- Inherited scale rules include anisotropic corrections that conflict with the organizer's preserved-aspect-ratio statement. Range clipping, texture fusion, the 0.9 FL correction and base model weights also remain inherited; some were tuned using the public leaderboard.
- Python 3.13 required a tested, narrow RapidOCR 1.4.4 metadata override, not an upstream support claim. A duplicate NumPy metadata warning was resolved using same-kernel runtime evidence and retained transparently in the run audit.
- Scalar held-out PA/FL/MT ground truth is not supplied by the competition; segmentation proxy validation cannot establish the final measurement error.
- Model fold construction and reported Dice come from the model authors, not a fresh training audit. No model is retrained here.
- If the sample is a two-row format example, infer the 309-image manifest from actual test files; if a full template exists, preserve its order and resolve filenames only by unique stem.

## Attribution and licenses

Adapted code: [prvsiyan, pinned V12/366298975](https://www.kaggle.com/code/prvsiyan/trying-some-stuff-with-ultrasound-geometry?scriptVersionId=366298975), based on [Mawra Musawwar](https://www.kaggle.com/code/mawramusawwar/umud-two-u-nets-texture-angle-lb-0-30652), [watehell](https://www.kaggle.com/code/watehell/ans-is-simple-physical-scale-u-net-geometry), and [Rishabh Bafna](https://www.kaggle.com/code/rishabhbafnaiiitd/muscle-architecture-in-ultrasound-data-kickstart). Observed-support rule: [auriclee](https://www.kaggle.com/code/auriclee/0-29922-a-small-geometry-fix).

Code is adapted from Apache-2.0 notebook sources; retain attribution. Fivefold weights are CC BY-NC 4.0; the original watehell pair is CC BY-NC-SA 4.0, as verified from the public dataset metadata. Competition terms apply separately. No weights or competition images are redistributed in this package.


See `RESEARCH_REVIEW.md` and `validation_and_provenance.json` for the completed checks, exact source comparisons and unresolved assumptions. No clinical accuracy or independent generalization claim is made.


### 1. Set up and verify inputs

In [2]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, platform, importlib.metadata as metadata
import random, time

# All task outputs stay inside this new directory. No raw inputs are modified.
INPUT = Path(os.environ.get("UMUD_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("UMUD_WORK", "/kaggle/working/umud_supported_depth_v1"))
WORK.mkdir(parents=True, exist_ok=True)
AUDIT = WORK / "audit"
AUDIT.mkdir(exist_ok=True)

# Pin the architecture package used by the checkpoint authors. Record every installed version.
if os.environ.get("UMUD_SKIP_INSTALL", "0") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", "segmentation-models-pytorch==0.5.0", "pyclipper>=1.2.0", "opencv-python>=4.5.1.48", "numpy>=1.19.5,<3", "six>=1.15.0", "Shapely>=1.7.1,!=2.0.4", "PyYAML", "Pillow", "onnxruntime>=1.20", "tqdm"], check=True)
    # Exact pure-Python OCRv4 wheel smoke-tested under Python 3.13.5.
    # Resolve dependencies normally; override only this wheel's upper Python metadata bound.
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", "--no-deps", "--ignore-requires-python", "rapidocr-onnxruntime==1.4.4"], check=True)

import numpy as np
import pandas as pd
import torch

SEED = 20261112
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_num_threads(min(4, os.cpu_count() or 1))

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def unique_file(name):
    hits = sorted({p.resolve() for p in INPUT.rglob(name) if p.is_file()})
    if len(hits) != 1:
        raise RuntimeError(f"Expected one {name}; found {len(hits)}. Attach only the named inputs.")
    return hits[0]

test_dirs = sorted({p.resolve() for p in INPUT.rglob("test_set_v2") if p.is_dir()})
if len(test_dirs) != 1:
    raise RuntimeError(f"Expected one test_set_v2 folder; found {len(test_dirs)}")
TEST_DIR = test_dirs[0]
files = sorted(p for p in TEST_DIR.iterdir() if p.suffix.lower() in {".tif", ".tiff", ".png"})
if len(files) != 309 or len({p.name for p in files}) != len(files):
    raise RuntimeError("Current competition input must have 309 uniquely named test images")

EXPECTED_FOLD_HASHES = {'apo_r34_f0.pt': 'de270ef650a57ab0fd01fbd3c19a58a064c016eccb8aaef22f1105de0389e04e', 'apo_r34_f1.pt': 'd6dd441dbbd32b4441da0a533ca88d6629e380c8bfa1e892fe4fc4b67d036960', 'apo_r34_f2.pt': '69713b5a03d2a7cb4e1f0d498d1161414ebf3b16d5cd7cd1698b7333b49af6e0', 'apo_r34_f3.pt': '5dfb0196d857ad45504a18829003b422a7e3782eb4d534db919fe09f1c7a8627', 'apo_r34_f4.pt': 'c76872cc5740260700f8c899b30a1881c96247ce362233522c064f6e9c400536', 'fasc_r34_f0.pt': '9842e2be8bf815de29822c0ff406ce08979822d2805683dae442353e8886256a', 'fasc_r34_f1.pt': 'd2a30df2136fc5015c39f5214ef7fd9e68e6d1d1dfffa8b433b780ece753d104', 'fasc_r34_f2.pt': 'eca288f3746dc673d2b26c130ab68fb0b33ed24a2ecb1b5445b6a46758336a4c', 'fasc_r34_f3.pt': '799a0b214b88d06bb34aa6efc8116c89d38ee6e985b6192f41e2e395548f372a', 'fasc_r34_f4.pt': '9a1faaf535c43fff2efa8155409aa30b1155403a71aa7db3424941fc2cc7c934'}
checkpoint_names = ["apo_resnet34_384x576_f0.pt", "fasc_resnet34_384x576_f0.pt"] + list(EXPECTED_FOLD_HASHES)
checkpoints = {name: unique_file(name) for name in checkpoint_names}
checkpoint_audit = []
for name, path in checkpoints.items():
    digest = sha256_file(path)
    if name in EXPECTED_FOLD_HASHES and digest != EXPECTED_FOLD_HASHES[name]:
        raise RuntimeError(f"Checkpoint checksum does not match the author's manifest: {name}")
    checkpoint_audit.append({"name": name, "bytes": path.stat().st_size, "sha256": digest,
                             "expected_hash_verified": name in EXPECTED_FOLD_HASHES})

# The original pair has no supplied hash manifest; observed hashes are recorded, not called verified.
pd.DataFrame(checkpoint_audit).to_csv(AUDIT / "checkpoint_hashes.csv", index=False)
image_audit = [{"image_id": p.name, "bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in files]
pd.DataFrame(image_audit).to_csv(AUDIT / "input_images.csv", index=False)

links = {WORK / "data" / "test_set_v2": TEST_DIR,
         WORK / "checkpoints": checkpoints["apo_resnet34_384x576_f0.pt"].parent,
         WORK / "checkpoints_ens": checkpoints["apo_r34_f0.pt"].parent}
for link, target in links.items():
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.exists() or link.is_symlink():
        if link.resolve() != target:
            raise RuntimeError(f"Existing link points to different input: {link}")
    else:
        link.symlink_to(target, target_is_directory=True)

versions = {d.metadata["Name"]: d.version for d in metadata.distributions() if d.metadata["Name"]}
(AUDIT / "environment.json").write_text(json.dumps({"python": sys.version, "platform": platform.platform(),
    "versions": versions, "cuda_available": torch.cuda.is_available(), "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "seed": SEED}, indent=2))
print("Inputs:", len(files), "images;", len(checkpoints), "checkpoints; GPU:", torch.cuda.is_available())


Inputs: 309 images; 12 checkpoints; GPU: True


### 2. Install the audited source modules

In [3]:
MODULES = {'umud/seg.py': '"""U-Net segmentation of aponeuroses and fascicles, with an on-disk cache of probability maps.\n\nTwo model sets:\n* tag "test" / "osf": single ResNet-34 U-Nets from checkpoints/ (dataset watehell/umud-r34-unet-weights),\n  input read with cv2 grayscale.\n* tag "test_ens": our own 5-fold ResNet-34 U-Net ensemble from checkpoints_ens/ (output of the Kaggle\n  notebook umud-seg-ensemble-r34), input read with PIL grayscale and maps quantised to uint8 exactly\n  as in that notebook.\n\nCheckpoints are dicts {"encoder", "state", "size"} for smp.Unet(in_channels=1, classes=1).\nInference uses a horizontal-flip TTA. Maps are cached as float16 so geometry experiments can be\nre-run in seconds without touching the networks again.\n"""\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport segmentation_models_pytorch as smp\nimport torch\nfrom PIL import Image\n\nROOT = Path(__file__).resolve().parents[1]\nCKPT = ROOT / "checkpoints"\nCKPT_ENS = ROOT / "checkpoints_ens"\nCACHE = ROOT / "cache"\nDEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nENS_FOLDS = 5\n\n\ndef read_gray(path):\n    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)\n    if g is None:\n        raise FileNotFoundError(path)\n    return g\n\n\ndef load_model(path):\n    c = torch.load(path, map_location="cpu", weights_only=True)\n    m = smp.Unet(c["encoder"], encoder_weights=None, in_channels=1, classes=1)\n    m.load_state_dict({k: v.float() for k, v in c["state"].items()})\n    return m.to(DEV).eval(), tuple(c["size"])\n\n\n@torch.no_grad()\ndef predict(models, size, gray):\n    """Mean of sigmoid(model(x)) and its h-flip over all models."""\n    oh, ow = size\n    x = cv2.resize(gray, (ow, oh), interpolation=cv2.INTER_AREA).astype(np.float32) / 255\n    x = torch.from_numpy(x)[None, None].to(DEV)\n    acc = 0\n    for m in models:\n        acc = acc + torch.sigmoid(m(x)) + torch.sigmoid(m(x.flip(-1))).flip(-1)\n    return (acc / (2 * len(models)))[0, 0].cpu().numpy()\n\n\ndef _model_sets(tag):\n    if tag.startswith("ens_f"):\n        f = int(tag[5:])\n        sets = []\n        for kind in ("apo", "fasc"):\n            m, size = load_model(CKPT_ENS / f"{kind}_r34_f{f}.pt")\n            sets.append(([m], size))\n        return sets\n    if tag == "test_ens":\n        sets = []\n        for kind in ("apo", "fasc"):\n            loaded = [load_model(CKPT_ENS / f"{kind}_r34_f{f}.pt") for f in range(ENS_FOLDS)]\n            sets.append(([m for m, _ in loaded], loaded[0][1]))\n        return sets\n    return [([m], s) for m, s in (load_model(CKPT / "apo_resnet34_384x576_f0.pt"),\n                                  load_model(CKPT / "fasc_resnet34_384x576_f0.pt"))]\n\n\ndef _infer(sets, tag, path):\n    if tag.startswith("ens_f"):\n        g = np.asarray(Image.open(path).convert("L"))\n        pa, pf = (predict(ms, size, g) for ms, size in sets)\n    elif tag == "test_ens":\n        g = np.asarray(Image.open(path).convert("L"))\n        pa, pf = (np.round(predict(ms, size, g) * 255) / 255 for ms, size in sets)\n    else:\n        g = read_gray(path)\n        pa, pf = (predict(ms, size, g) for ms, size in sets)\n    return pa, pf, g.shape\n\n\ndef cached_maps(files, tag):\n    """Return {name: (p_apo, p_fasc, (H0, W0))}, computing and caching missing entries."""\n    out_dir = CACHE / tag\n    out_dir.mkdir(parents=True, exist_ok=True)\n    sets = None\n    res = {}\n    for i, f in enumerate(files):\n        f = Path(f)\n        npz = out_dir / (f.name + ".npz")\n        if not npz.exists():\n            if sets is None:\n                if DEV.type == "cpu":\n                    torch.set_num_threads(4)\n                sets = _model_sets(tag)\n            pa, pf, shape = _infer(sets, tag, f)\n            np.savez_compressed(npz, apo=pa.astype(np.float16), fasc=pf.astype(np.float16), shape=np.array(shape))\n            if (i + 1) % 50 == 0:\n                print(f"{tag}: {i + 1}/{len(files)}", flush=True)\n        z = np.load(npz)\n        for key in ("apo", "fasc"):\n            a = z[key]\n            if a.ndim != 2 or not np.isfinite(a).all() or a.min() < 0 or a.max() > 1:\n                raise ValueError(f"Invalid cached probability map: {npz}, {key}")\n        res[f.name] = (z["apo"].astype(np.float32), z["fasc"].astype(np.float32), tuple(z["shape"]))\n    return res\n\n\nif __name__ == "__main__":\n    import sys\n    import time\n\n    t = time.time()\n    test = sorted((ROOT / "data" / "test_set_v2").glob("IMG_*"))\n    osf = sorted((ROOT / "data" / "external" / "osf_arch" / "benchmark_dataset_architecture_v0.1.0").glob("im_*.tif"))\n    which = sys.argv[1:] or ["osf", "test"]\n    if "osf" in which:\n        cached_maps(osf, "osf")\n    for tag in ("test", "test_ens"):\n        if tag in which:\n            cached_maps(test, tag)\n    print(f"done in {time.time() - t:.0f}s")\n\ndef derive_ens(files, folds=range(ENS_FOLDS)):\n    """Rebuild the \'test_ens\' cache (mean of the five fold maps, uint8-quantised as in the 80-minute single-pass run)."""\n    out_dir = CACHE / "test_ens"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    for f in files:\n        zs = [np.load(CACHE / f"ens_f{k}" / (Path(f).name + ".npz")) for k in folds]\n        pa = np.mean([z["apo"].astype(np.float32) for z in zs], 0)\n        pf = np.mean([z["fasc"].astype(np.float32) for z in zs], 0)\n        pa, pf = np.round(pa * 255) / 255, np.round(pf * 255) / 255\n        np.savez_compressed(out_dir / (Path(f).name + ".npz"), apo=pa.astype(np.float16),\n                            fasc=pf.astype(np.float16), shape=zs[0]["shape"])\n', 'umud/scale.py': '"""Per-image physical scale (pixels per mm along x and y) for the UMUD test images.\n\nThe test frames are screenshots from three devices, most resized non-uniformly to 1200x800,\nso x and y need separate scales:\n\n| layout   | sy (px/mm vertical)                        | sx (px/mm horizontal)                  |\n|----------|--------------------------------------------|----------------------------------------|\n| siemens  | depth-ruler span / OCR depth label          | sy * 1.125 (1152x864 -> 1200x800)      |\n| telemed  | left ruler, 10 mm per tick                  | bottom ruler, 10 mm per tick           |\n| crop     | = sx                                       | bottom ruler, 10 mm per tick           |\n| lumify   | depth-ruler span / OCR depth label          | sy * width/height of the round P icon  |\n\nMissing OCR values are borrowed from frames with an identical ruler, and outliers are snapped to\ngroup medians (same device + depth). Results are cached to cache/scale_test.csv.\n"""\nimport re\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nROOT = Path(__file__).resolve().parents[1]\nSIEMENS_ASPECT = (1200 / 1152) / (800 / 864)\n_ocr = None\n\n\ndef ocr_text(img, scale=3):\n    global _ocr\n    if _ocr is None:\n        from rapidocr_onnxruntime import RapidOCR\n        _ocr = RapidOCR()\n    big = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)\n    res, _ = _ocr(big)\n    return " ".join(r[1] for r in (res or []))\n\n\ndef parse_depth_mm(text):\n    m = re.search(r"(\\d+(?:[.,]\\d+)?)\\s*(cm|mm)", text)\n    if not m:\n        return np.nan\n    v = float(m.group(1).replace(",", "."))\n    return v * 10 if m.group(2) == "cm" else v\n\n\ndef tick_positions(profile, min_gap=12, max_width=6):\n    """Centres of thin bright peaks in a 1-D intensity profile."""\n    p = profile.astype(np.float32)\n    on = p > max(p.max() * 0.5, 40)\n    pos, i = [], 0\n    while i < len(p):\n        if on[i]:\n            j = i\n            while j < len(p) and on[j]:\n                j += 1\n            if j - i <= max_width:\n                pos.append((i + j - 1) / 2)\n            i = j\n        else:\n            i += 1\n    keep = []\n    for q in pos:\n        if not keep or q - keep[-1] >= min_gap:\n            keep.append(q)\n    return np.array(keep)\n\n\ndef regular_train(pos, tol=0.06, max_skip=3):\n    """Longest evenly spaced run of ticks (missed ticks allowed) -> (spacing, n_intervals, first, last)."""\n    best, key = (np.nan, 0, np.nan, np.nan), (0, 0)\n    if len(pos) < 3:\n        return best\n    for i in range(len(pos) - 1):\n        s = pos[i + 1] - pos[i]\n        for k in range(1, max_skip + 1):\n            sk = s / k\n            steps, last, hits = k, pos[i + 1], 2\n            for q in pos[i + 2:]:\n                m = round((q - last) / sk)\n                if 1 <= m <= max_skip and abs(q - last - m * sk) < tol * sk * m:\n                    steps += m\n                    last = q\n                    hits += 1\n                elif q - last > max_skip * sk * 1.5:\n                    break\n            if hits >= 3 and hits >= 0.6 * (steps + 1) and (steps, hits) > key:\n                best, key = ((last - pos[i]) / steps, steps, pos[i], last), (steps, hits)\n    return best\n\n\ndef best_ruler(gray, axis, band, span):\n    """Scan the rows/columns in `band` for the longest regular tick train within `span`."""\n    best = (np.nan, 0, np.nan, np.nan)\n    lo, hi = span\n    for k in range(*band):\n        prof = gray[lo:hi, k] if axis == "y" else gray[k, lo:hi]\n        t = regular_train(tick_positions(prof))\n        if t[1] > best[1]:\n            best = (t[0], t[1], t[2] + lo, t[3] + lo)\n    return best\n\n\ndef layout(path, gray):\n    h, w = gray.shape\n    if path.suffix == ".png":\n        return "lumify"\n    if (w, h) in ((1088, 644), (1200, 800)):\n        side = np.median(gray[100:h - 20, 90:140])  # Telemed screenshots have mid-grey side bars\n        return "telemed" if 35 < side < 80 else "siemens"\n    return "crop"\n\n\ndef lumify_icon_ratio(gray):\n    """Width/height of the round \'P\' icon, which the non-uniform resize turned into an ellipse."""\n    colmean = gray[150:600].astype(float).mean(0)\n    x0 = int(np.argmax(colmean[200:700] > 25)) + 200\n    roi = gray[60:250, max(x0 - 90, 0):x0 - 3]\n    n, lab, st, _ = cv2.connectedComponentsWithStats((roi > 20).astype(np.uint8))\n    if n < 2:\n        return np.nan\n    i = 1 + np.argmax(st[1:, 4])\n    cs, _ = cv2.findContours((lab == i).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n    _, (a, b), ang = cv2.fitEllipse(cs[0])\n    wa, ha = (a, b) if (ang < 45 or ang > 135) else (b, a)\n    return wa / ha\n\n\ndef measure_one(path):\n    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)\n    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)\n    h, w = gray.shape\n    lay = layout(path, gray)\n    r = dict(image_id=path.name, layout=lay, w=w, h=h, sx=np.nan, sy=np.nan, aspect=np.nan,\n             depth_mm=np.nan, tick_mm=np.nan, ny=0, nx=0, span=np.nan, ocr="", depth_src="ocr")\n    if lay == "siemens":\n        r["ocr"] = ocr_text(bgr[90:h - 60, 960:1100])\n        _, n, a, b = best_ruler(gray, "y", (w - 80, w), (40, h - 40))\n        r.update(depth_mm=parse_depth_mm(r["ocr"]), ny=n, span=b - a, aspect=SIEMENS_ASPECT)\n    elif lay == "telemed":\n        r["ocr"] = ocr_text(bgr[40:140, w - 110:w])\n        r["depth_mm"] = parse_depth_mm(r["ocr"])\n        band = (0, 20) if w == 1088 else (40, 110)\n        s, n, _, _ = best_ruler(gray, "y", band, (0, h))\n        r.update(sy=s / 10, ny=n, tick_mm=10.0)\n        s, n, _, _ = best_ruler(gray, "x", (h - 12, h), (130, w - 130))\n        r.update(sx=s / 10, nx=n)\n        r["aspect"] = r["sx"] / r["sy"]\n    elif lay == "crop":\n        s, n, _, _ = best_ruler(gray, "x", (h - 8, h), (0, w))\n        r.update(sx=s / 10, sy=s / 10, nx=n, tick_mm=10.0, aspect=1.0)\n    else:  # lumify\n        r["ocr"] = ocr_text(bgr[60:600, w - 120:w])\n        _, n, a, b = best_ruler(gray, "y", (0, 30), (90, h - 40))\n        r.update(depth_mm=parse_depth_mm(r["ocr"]), ny=n, span=b - a, aspect=lumify_icon_ratio(gray))\n    return r\n\n\ndef _fill_depth_rulers(df):\n    for lay in ("siemens", "lumify"):\n        m = df.layout == lay\n        key = lambda g: list(zip(g.ny, (g.span / 4).round()))\n        ok = m & df.depth_mm.notna()\n        lut = {}\n        for k, d in zip(key(df[ok]), df.depth_mm[ok]):\n            lut.setdefault(k, []).append(d)\n        miss = m & df.depth_mm.isna()\n        for i, k in zip(df.index[miss], key(df[miss])):\n            vals = lut.get(k, [])\n            if vals and len(set(vals)) == 1:\n                df.loc[i, ["depth_mm", "depth_src"]] = vals[0], "lut"\n        df.loc[m, "sy"] = df.span[m] / df.depth_mm[m]\n        df.loc[m, "sx"] = df.sy[m] * df.aspect[m]\n        df.loc[m, "tick_mm"] = df.depth_mm[m] / df.ny[m]\n    return df\n\n\ndef _fill_crops(df, tol=0.05):\n    crop = df.layout == "crop"\n    grp = (df.h / 10).round()\n    med = df[crop].groupby(grp[crop]).sx.median()\n    for i in df.index[crop]:\n        m = med[grp[i]]\n        if not abs(df.sx[i] / m - 1) < tol:\n            df.loc[i, ["sx", "sy"]] = m\n    return df\n\n\ndef _snap_odd_ticks(df, ok_ticks=(2.0, 2.5, 5.0, 10.0)):\n    std = df.tick_mm.round(2).isin(ok_ticks)\n    med = df[std].groupby(["layout", "depth_mm"])[["sx", "sy"]].median()\n    for i in df.index[~std & df.depth_mm.notna()]:\n        k = (df.layout[i], df.depth_mm[i])\n        if k in med.index:\n            df.loc[i, ["sx", "sy"]] = med.loc[k].values\n    return df\n\n\ndef _snap_device_aspect(df, tol=0.03):\n    tm = df.layout == "telemed"\n    med = df[tm].groupby(["w", "h"]).aspect.median()\n    for i in df.index[tm]:\n        m = med[(df.w[i], df.h[i])]\n        if not abs(df.aspect[i] / m - 1) < tol:\n            df.loc[i, "sx"] = df.sy[i] * m\n            df.loc[i, "aspect"] = m\n    return df\n\n\ndef test_scales(files, refresh=False):\n    out = ROOT / "cache" / "scale_test.csv"\n    if out.exists() and not refresh:\n        return pd.read_csv(out)\n    df = pd.DataFrame([measure_one(Path(p)) for p in files])\n    df = _snap_device_aspect(_snap_odd_ticks(_fill_crops(_fill_depth_rulers(df))))\n    out.parent.mkdir(exist_ok=True)\n    df.to_csv(out, index=False)\n    return df\n\n\nif __name__ == "__main__":\n    files = sorted((ROOT / "data" / "test_set_v2").glob("IMG_*"))\n    df = test_scales(files, refresh=True)\n    print(df.groupby("layout").agg(n=("sx", "size"), sx=("sx", "median"), sy=("sy", "median"),\n                                   aspect=("aspect", "median"), missing=("sx", lambda s: s.isna().sum())).round(3))\n    print("depth source:", df.depth_src.value_counts().to_dict())', 'umud/geometry.py': '"""PA / FL / MT from aponeurosis and fascicle probability maps, in millimetres.\n\nHost protocol (forum answers by the organizers):\n* MT: three *vertical* image lines (left / middle / right) between the aponeuroses, averaged.\n* PA: angle between fascicle and the slope of the *deep* aponeurosis, at fascicle insertions.\n* FL: three fascicles chosen to minimise extrapolation, extended linearly to both aponeuroses.\n\nAll coordinates are physical: X = x * mx, Y = y * my (mm per probability-map pixel), so\nnon-square pixels are handled. Every choice that is not fixed by the protocol is a key in\n`DEFAULT`, so variants can be compared on the expert-labelled OSF benchmark.\n"""\nimport cv2\nimport numpy as np\n\nDEFAULT = dict(\n    apo_thr=0.5,\n    mt_mode="inner_vertical",   \n    mt_xs=(0.2, 0.5, 0.8),      \n    pa_mode="depth_fit",        \n    pa_at=0.9,                  \n    fl_mode="extend_inner",     \n    fl_select="weighted",       \n    fl_factor=1.0,\n)\nFALLBACK = dict(pa_deg=15.0, fl_mm=70.0, mt_mm=20.0)\n\n\ndef _column_profile(mask_comp, prob, cols):\n    """Per column of one component: weighted centre, top edge and bottom edge (map pixels)."""\n    ys, xs = np.nonzero(mask_comp)\n    cy, top, bot = [], [], []\n    for c in cols:\n        yc = ys[xs == c]\n        cy.append(np.average(yc, weights=prob[yc, c]))\n        top.append(yc.min() - 0.5)\n        bot.append(yc.max() + 0.5)\n    return np.array(cy), np.array(top), np.array(bot)\n\n\ndef aponeuroses(p_apo, mx, my, thr=0.5, merge_mm=4.0, min_gap_mm=6.0, min_rel_extent=0.35):\n    """Superficial and deep aponeurosis as dicts of quadratic fits (Y in mm as a function of X in mm).\n\n    Each band has fits for its centreline ("c"), upper edge ("top") and lower edge ("bot").\n    Components less than `merge_mm` apart in depth are merged (fragmented / double lines).\n    """\n    h, w = p_apo.shape\n    n, lab, st, _ = cv2.connectedComponentsWithStats((p_apo > thr).astype(np.uint8), connectivity=8)\n    comps = []\n    for i in range(1, n):\n        if st[i, 2] < 0.03 * w:\n            continue\n        m = lab == i\n        cols = np.unique(np.nonzero(m)[1])\n        cy, top, bot = _column_profile(m, p_apo, cols)\n        comps.append(dict(X=cols * mx, c=cy * my, top=top * my, bot=bot * my))\n    if not comps:\n        return None\n    xc = np.median(np.concatenate([c["X"] for c in comps]))\n    depth = [np.polyval(np.polyfit(c["X"], c["c"], 1), xc) if len(c["X"]) > 1 else c["c"].mean() for c in comps]\n    bands = []\n    for k in np.argsort(depth):\n        if bands and depth[k] - bands[-1]["d"] < merge_mm:\n            b = bands[-1]\n            for key in ("X", "c", "top", "bot"):\n                b[key] = np.concatenate([b[key], comps[k][key]])\n        else:\n            bands.append(dict(d=depth[k], **comps[k]))\n    ext = np.array([b["X"].max() - b["X"].min() for b in bands])\n    bands = [b for b, e in zip(bands, ext) if e >= min_rel_extent * ext.max()]\n    out = []\n    for b in bands:\n        if out and b["d"] - out[-1]["d"] < min_gap_mm:\n            continue\n        deg = 2 if b["X"].max() - b["X"].min() > 15 else 1\n        b["fit"] = {k: np.polyfit(b["X"], b[k], deg) for k in ("c", "top", "bot")}\n        b["lo"], b["hi"] = b["X"].min(), b["X"].max()\n        out.append(b)\n        if len(out) == 2:\n            break\n    return out if len(out) == 2 else None\n\n\ndef _shared_span(sup, deep):\n    lo, hi = max(sup["lo"], deep["lo"]), min(sup["hi"], deep["hi"])\n    if hi - lo < 3:\n        lo, hi = min(sup["lo"], deep["lo"]), max(sup["hi"], deep["hi"])\n    return lo, hi\n\n\ndef muscle_thickness(sup, deep, mode, xs_rel):\n    lo, hi = _shared_span(sup, deep)\n    X = lo + np.array(xs_rel) * (hi - lo)\n    if mode == "inner_vertical":\n        d = np.polyval(deep["fit"]["top"], X) - np.polyval(sup["fit"]["bot"], X)\n    elif mode == "centre_vertical":\n        d = np.polyval(deep["fit"]["c"], X) - np.polyval(sup["fit"]["c"], X)\n    elif mode == "centre_minus_half":\n        th = lambda b: np.median(b["bot"] - b["top"])\n        d = np.polyval(deep["fit"]["c"], X) - np.polyval(sup["fit"]["c"], X) - (th(sup) + th(deep)) / 2\n    else:\n        raise ValueError(mode)\n    return float(np.mean(d))\n\n\ndef fascicle_segments(p_fasc, mx, my, thr=0.5, min_len_mm=4.0):\n    """Long, thin connected components -> (cx, cy, angle_deg, length_mm) in physical units."""\n    n, lab, st, _ = cv2.connectedComponentsWithStats((p_fasc > thr).astype(np.uint8), connectivity=8)\n    segs = []\n    for i in range(1, n):\n        if st[i, 4] < 6:\n            continue\n        ys, xs = np.nonzero(lab == i)\n        P = np.stack([xs * mx, ys * my], 1)\n        c = P.mean(0)\n        _, s, vt = np.linalg.svd(P - c, full_matrices=False)\n        proj = (P - c) @ vt[0]\n        L = proj.max() - proj.min()\n        if L < min_len_mm or s[0] < 3 * (s[1] + 1e-6):\n            continue\n        ang = (np.degrees(np.arctan2(vt[0][1], vt[0][0])) + 90) % 180 - 90\n        segs.append((c[0], c[1], ang, L))\n    return segs\n\n\ndef _slope_deg(poly, X):\n    return float(np.degrees(np.arctan(np.polyval(np.polyder(poly), X))))\n\n\ndef _intersect(cx, cy, ang, poly):\n    """Intersection of the line through (cx, cy) at `ang` with the polynomial curve nearest to cx."""\n    k = np.tan(np.radians(ang))\n    q = np.array(poly, dtype=float).copy()\n    q[-1] -= cy - k * cx\n    q[-2] -= k\n    r = np.roots(q)\n    r = r[np.isreal(r)].real\n    if len(r) == 0:\n        return None\n    X = r[np.argmin(np.abs(r - cx))]\n    return np.array([X, cy + k * (X - cx)])\n\n\ndef _wmedian(v, w):\n    o = np.argsort(v)\n    cw = np.cumsum(w[o])\n    return float(v[o][np.searchsorted(cw, cw[-1] / 2)])\n\n\ndef measure(p_apo, p_fasc, mx, my, **kw):\n    """Measure one image. Returns dict with pa_deg, fl_mm, mt_mm, ok and per-segment data."""\n    P = {**DEFAULT, **kw}\n    out = dict(FALLBACK, ok=False, segs=np.zeros((0, 4)))\n    apo = aponeuroses(p_apo, mx, my, thr=P["apo_thr"])\n    if apo is None:\n        return out\n    sup, deep = apo\n    out["apo"] = apo\n    out["mt_mm"] = muscle_thickness(sup, deep, P["mt_mode"], P["mt_xs"])\n    inner = P["fl_mode"] != "extend_centre"\n    sup_line = sup["fit"]["bot"] if inner else sup["fit"]["c"]\n    deep_line = deep["fit"]["top"] if inner else deep["fit"]["c"]\n\n    rows = []  \n    for cx, cy, ang, L in fascicle_segments(p_fasc, mx, my):\n        ys, yd = np.polyval(sup["fit"]["c"], cx), np.polyval(deep["fit"]["c"], cx)\n        if not (ys - 2 < cy < yd + 2):\n            continue\n        pa = abs(((ang - _slope_deg(deep["fit"]["c"], np.clip(cx, deep["lo"], deep["hi"]))) + 90) % 180 - 90)\n        if not (2 < pa < 60):\n            continue\n        p1, p2 = _intersect(cx, cy, ang, sup_line), _intersect(cx, cy, ang, deep_line)\n        fl = np.linalg.norm(p1 - p2) if p1 is not None and p2 is not None else np.nan\n        rows.append(((cy - ys) / max(yd - ys, 1e-6), pa, L, fl))\n    if not rows:\n        return out\n    S = np.array(rows)\n    out["segs"] = S\n    out["ok"] = True\n\n    pa = None\n    if P["pa_mode"] == "depth_fit":\n        pa = pa_depth_fit(S, P["pa_at"])\n    elif P["pa_mode"] == "deep_half":\n        deep_rows = S[S[:, 0] > 0.5]\n        if len(deep_rows) >= 2:\n            pa = _wmedian(deep_rows[:, 1], deep_rows[:, 2])\n    out["pa_deg"] = pa if pa is not None else _wmedian(S[:, 1], S[:, 2])\n\n    fl_sin = out["mt_mm"] / np.sin(np.radians(max(out["pa_deg"], 3.0)))\n    good = np.isfinite(S[:, 3]) & (S[:, 3] > 10) & (S[:, 3] < 300)\n    if not good.any():\n        fl_ext = fl_sin\n    elif P["fl_select"] == "top3":\n        g = S[good]\n        fl_ext = float(np.mean(g[np.argsort(-g[:, 2])[:3], 3]))\n    elif P["fl_select"] == "visible":\n        fl_ext = _wmedian(S[good, 3], S[good, 2] ** 2 / S[good, 3])\n    else:\n        fl_ext = _wmedian(S[good, 3], S[good, 2])\n    if P["fl_mode"] == "mt_over_sin":\n        fl = fl_sin\n    elif P["fl_mode"] == "avg":\n        fl = 0.5 * (fl_ext + fl_sin)\n    else:\n        fl = fl_ext\n    out["fl_mm"] = fl * P["fl_factor"]\n    return out\n\n\ndef pa_depth_fit(S, at, min_n=5):\n    """Fit PA against depth and evaluate only within observed segment support."""\n    S = np.asarray(S, dtype=float)\n    if S.ndim != 2 or S.shape[1] != 4:\n        raise ValueError("Expected segment columns: relative_depth, PA, visible_length, extended_length")\n    if not np.isfinite(S[:, :3]).all() or (S[:, 2] <= 0).any():\n        raise ValueError("Invalid depth/angle/visible-length measurement")\n    if len(S) < min_n or np.ptp(S[:, 0]) < 0.2:\n        return None\n    # Conservative change: use the observed minimum and maximum, keeping at=0.9.\n    # Applied identically to each image and each content-similar frame group.\n    at = float(np.clip(at, S[:, 0].min(), S[:, 0].max()))\n    return float(np.polyval(np.polyfit(S[:, 0], S[:, 1], 1, w=np.sqrt(S[:, 2])), at))', 'umud/texture.py': '"""Pennation angle from raw image texture, independent of the fascicle segmentation model.\n\nThe image is resampled to isotropic pixels (r px/mm) so angles are physical. Inside the deep part of\nthe muscle belly (between the inner aponeurosis edges found by the aponeurosis model), a high-pass\nfiltered image is projected onto candidate directions: when the direction matches the fascicles,\nthe bright fascicle lines fall into the same projection bins and the variance of the bin means peaks\n(a discrete Radon-transform maximum). Near-horizontal directions are excluded because speckle and\nresidual aponeurosis brightness are elongated laterally.\n"""\nimport cv2\nimport numpy as np\n\nfrom umud.geometry import _slope_deg\n\n\ndef texture_angle(gray, sx, sy, apo, r=4.0, depth_lo=0.45, depth_hi=0.92, margin_mm=1.0,\n                  angles=np.arange(-50, 50.5, 0.5), min_abs=4.0):\n    """Return (fascicle angle in image coords, PA vs deep aponeurosis) in degrees, or (nan, nan)."""\n    sup, deep = apo\n    H0, W0 = gray.shape\n    iso = cv2.resize(gray, (max(int(W0 * r / sx), 8), max(int(H0 * r / sy), 8)), interpolation=cv2.INTER_AREA)\n    iso = iso.astype(np.float32)\n    hp = iso - cv2.GaussianBlur(iso, (0, 0), 2.0 * r)\n    h, w = hp.shape\n    X = (np.arange(w) + 0.5) / r\n    lo, hi = max(sup["lo"], deep["lo"]), min(sup["hi"], deep["hi"])\n    ytop = np.polyval(sup["fit"]["bot"], X) + margin_mm\n    ybot = np.polyval(deep["fit"]["top"], X) - margin_mm\n    y0 = ytop + depth_lo * (ybot - ytop)\n    y1 = ytop + depth_hi * (ybot - ytop)\n    Y = (np.arange(h)[:, None] + 0.5) / r\n    roi = (Y >= y0[None]) & (Y <= y1[None]) & (X[None] >= lo) & (X[None] <= hi) & (ybot > ytop)[None]\n    if roi.sum() < 400:\n        return np.nan, np.nan\n    yy, xx = np.nonzero(roi)\n    v = hp[yy, xx]\n    v = (v - v.mean()) / (v.std() + 1e-6)\n    best, best_a = -1, np.nan\n    for a in angles:\n        if abs(a) < min_abs:\n            continue\n        t = np.radians(a)\n        b = np.round(-xx * np.sin(t) + yy * np.cos(t)).astype(np.int64)\n        b -= b.min()\n        n = np.bincount(b)\n        s = np.bincount(b, weights=v)\n        ok = n >= 15\n        if ok.sum() < 5:\n            continue\n        mu = s[ok] / n[ok]\n        score = float(np.average(mu ** 2, weights=n[ok]))\n        if score > best:\n            best, best_a = score, a\n    if not np.isfinite(best_a):\n        return np.nan, np.nan\n    xc = np.clip((lo + hi) / 2, deep["lo"], deep["hi"])\n    pa = abs(((best_a - _slope_deg(deep["fit"]["c"], xc)) + 90) % 180 - 90)\n    return float(best_a), float(pa)', 'umud/make_sub.py': '"""Build a submission CSV from cached segmentation maps + per-image scales.\n\nUsage:  python -m umud.make_sub NAME [key=value ...]\ne.g.    python -m umud.make_sub v1 pa_at=0.9\nWrites submissions/NAME.csv and prints per-device summaries.\n"""\nimport sys\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nfrom umud import geometry, scale, seg, texture\n\nROOT = Path(__file__).resolve().parents[1]\nRANGES = {"pa_deg": (5, 45), "fl_mm": (30, 200), "mt_mm": (10, 50)}\n\n\ndef dhash(g, n=16):\n    s = cv2.resize(g, (n + 1, n), interpolation=cv2.INTER_AREA).astype(np.int16)\n    return (s[:, 1:] > s[:, :-1]).ravel()\n\n\ndef video_runs(files, max_dist=12):\n    """Group near-identical frames (the 5-frame video runs) with dHash + union-find."""\n    hs = np.array([dhash(seg.read_gray(f)) for f in files], dtype=np.uint8)\n    parent = list(range(len(hs)))\n\n    def find(a):\n        while parent[a] != a:\n            parent[a] = parent[parent[a]]\n            a = parent[a]\n        return a\n\n    for i in range(len(hs)):\n        for j in np.nonzero((hs[i + 1:] != hs[i]).sum(1) <= max_dist)[0] + i + 1:\n            parent[find(j)] = find(i)\n    return np.array([find(i) for i in range(len(hs))])\n\n\ndef build(params, pool_runs=True, maps_tag="test"):\n    params = dict(params)\n    tex_w = float(params.pop("tex_w", 0.0))\n    fl_sin_w = float(params.pop("fl_sin_w", 0.0))\n    files = sorted((ROOT / "data" / "test_set_v2").glob("IMG_*"))\n    maps = seg.cached_maps(files, maps_tag)\n    sc = scale.test_scales(files).set_index("image_id")\n    rows, segs = [], []\n    for f in files:\n        pa_map, pf_map, (H0, W0) = maps[f.name]\n        h, w = pa_map.shape\n        sx, sy = sc.sx[f.name], sc.sy[f.name]\n        m = geometry.measure(pa_map, pf_map, mx=(W0 / w) / sx, my=(H0 / h) / sy, **params)\n        segs.append(m.pop("segs"))\n        apo = m.pop("apo", None)\n        m["pa_tex"] = np.nan\n        if tex_w > 0 and apo is not None:\n            m["pa_tex"] = texture.texture_angle(seg.read_gray(f), sx, sy, apo)[1]\n        rows.append(dict(image_id=f.name, layout=sc.layout[f.name], **m))\n    df = pd.DataFrame(rows)\n\n    runs_file = ROOT / "cache" / "video_runs.npy"\n    if not runs_file.exists():\n        np.save(runs_file, video_runs(files))\n    df["run"] = np.load(runs_file)\n\n    P = {**geometry.DEFAULT, **params}\n    if pool_runs and P["pa_mode"] in ("depth_fit", "deep_half"):\n        for _, idx in df.groupby("run").groups.items():\n            S = np.concatenate([segs[i] for i in idx])\n            v = None\n            if P["pa_mode"] == "depth_fit" and len(S):\n                v = geometry.pa_depth_fit(S, P["pa_at"])\n            elif len(S):\n                deep = S[S[:, 0] > 0.5]\n                if len(deep) >= 2:\n                    v = geometry._wmedian(deep[:, 1], deep[:, 2])\n            if v is not None:\n                df.loc[idx, "pa_deg"] = v\n    if tex_w > 0:\n        tex = df.groupby("run").pa_tex.transform("median") if pool_runs else df.pa_tex\n        df["pa_deg"] = np.where(tex.notna(), (1 - tex_w) * df.pa_deg + tex_w * tex, df.pa_deg)\n    if fl_sin_w > 0:\n        fl_sin = df.mt_mm / np.sin(np.radians(df.pa_deg.clip(5, 45)))\n        df["fl_mm"] = (1 - fl_sin_w) * df.fl_mm + fl_sin_w * fl_sin\n    if not np.isfinite(df[list(RANGES)].to_numpy(dtype=float)).all():\n        raise ValueError("Non-finite geometry output before range clipping")\n    for c, (lo, hi) in RANGES.items():\n        df[c] = df[c].clip(lo, hi)\n    return df\n\n\ndef build_avg(params, tags):\n    """Measure on each map set separately, then average the measurements.\n\n    Averaging the probability maps instead distorts the thin fascicle lines and the aponeurosis\n    band thickness the geometry relies on (public LB 0.376 vs 0.319).\n    """\n    runs = [build(params, maps_tag=t) for t in tags]\n    df = runs[0]\n    for c in RANGES:\n        df[c] = np.mean([r[c].values for r in runs], axis=0)\n    return df\n\n\ndef main():\n    name = sys.argv[1]\n    params = {}\n    for a in sys.argv[2:]:\n        k, v = a.split("=")\n        try:\n            v = float(v)\n        except ValueError:\n            pass\n        params[k] = v\n    df = build_avg(params, str(params.pop("maps", "test")).split("+"))\n    out = ROOT / "submissions" / f"{name}.csv"\n    out.parent.mkdir(exist_ok=True)\n    df[["image_id", "pa_deg", "fl_mm", "mt_mm"]].round(3).to_csv(out, index=False)\n    print(df.groupby("layout")[["pa_deg", "fl_mm", "mt_mm"]].median().round(2))\n    print("fallback rows:", int((~df.ok).sum()), "| video runs >1:", int((df.run.value_counts() > 1).sum()))\n    print("wrote", out)\n\n\nif __name__ == "__main__":\n    main()'}
for relative, source in MODULES.items():
    destination = WORK / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")
(WORK / "umud" / "__init__.py").write_text("")
sys.path.insert(0, str(WORK))

config = {
    "method": "V12 image inference + observed min/max depth support at 0.9",
    "upstream_script_version": 366298975,
    "upstream_snapshot_sha256": "4a9f1e2fd3fa86b0cfca08daeb5fa2cdaadeb8358e75db6d348e93189f3c7464",
    "module_hashes": {name: hashlib.sha256(source.encode()).hexdigest() for name, source in MODULES.items()},
    "checkpoint_hashes": {row["name"]: row["sha256"] for row in checkpoint_audit},
    "images": image_audit,
    "dependencies": {name: metadata.version(name) for name in
                     ["torch", "torchvision", "segmentation-models-pytorch", "rapidocr-onnxruntime", "numpy", "opencv-python", "onnxruntime"]},
}
cache_guard = WORK / "cache_identity.json"
if cache_guard.exists() and json.loads(cache_guard.read_text()) != config:
    raise RuntimeError("Inputs, source or dependencies changed. Use a fresh UMUD_WORK directory.")
cache_guard.write_text(json.dumps(config, indent=2))
from umud import geometry, seg, scale, make_sub
print("Audited modules loaded; all cached maps and measurements will be retained.")


Audited modules loaded; all cached maps and measurements will be retained.


### 3. Test geometry, establish output order and validate physical scales

In [4]:
def synthetic_checks():
    depths = np.linspace(.2, .6, 5)
    segments = np.column_stack([depths, 20 + 10 * depths, np.ones(5) * 8, np.ones(5) * 60])
    assert abs(geometry.pa_depth_fit(segments, .9) - 26) < 1e-9
    assert abs(geometry.pa_depth_fit(segments, .4) - 24) < 1e-9
    assert geometry.pa_depth_fit(segments[:4], .9) is None
    narrow = segments.copy(); narrow[:, 0] = np.linspace(.3, .35, 5)
    assert geometry.pa_depth_fit(narrow, .9) is None
    invalid = segments.copy(); invalid[0, 2] = 0
    try:
        geometry.pa_depth_fit(invalid, .9)
    except ValueError:
        pass
    else:
        raise AssertionError("Zero-length segment was accepted")
    # A line at 30 degrees intersecting horizontal inner boundaries 20 mm apart has FL 40 mm.
    p1 = geometry._intersect(0, 10, 30, np.array([0., 0.]))
    p2 = geometry._intersect(0, 10, 30, np.array([0., 20.]))
    assert abs(np.linalg.norm(p2 - p1) - 40.) < 1e-8
    return {"bounded_extrapolation": True, "unchanged_interpolation": True,
            "weak_support_abstention": True, "invalid_segment_rejection": True, "physical_intersection": True}

(AUDIT / "synthetic_checks.json").write_text(json.dumps(synthetic_checks(), indent=2))

sample_files = sorted(INPUT.rglob("sample_submission.csv"))
if len(sample_files) != 1:
    raise RuntimeError("Expected one official sample_submission.csv")
sample = pd.read_csv(sample_files[0], sep=None, engine="python", encoding="utf-8-sig")
if list(sample.columns) != ["image_id", "pa_deg", "fl_mm", "mt_mm"] or sample.image_id.duplicated().any():
    raise RuntimeError("Unrecognized sample submission schema or duplicate IDs")
actual = {p.name: p for p in files}
by_stem = {p.stem: p.name for p in files}
if len(by_stem) != len(files):
    raise RuntimeError("Image stems are ambiguous")
if len(sample) == len(files):
    output_ids = [by_stem.get(Path(str(i)).stem) for i in sample.image_id]
    if None in output_ids or len(set(output_ids)) != len(files):
        raise RuntimeError("Full sample does not match actual test image IDs")
    order_reason = "Official full sample order; actual test extensions resolved uniquely by image stem"
else:
    # The official sample can be only a two-row format example, not an image manifest.
    if len(sample) != 2 or not all(Path(str(i)).stem in by_stem for i in sample.image_id):
        raise RuntimeError("Unexpected incomplete sample; review before exporting")
    output_ids = [p.name for p in files]
    order_reason = "Official two-row format example; all actual 309 test filenames sorted lexicographically"
(AUDIT / "submission_order.json").write_text(json.dumps({"sample_rows": len(sample),
    "reason": order_reason, "output_ids": output_ids}, indent=2))

# Calibrate once and stop if any physical conversion is invalid.
calibration = scale.test_scales(files)
if set(calibration.image_id) != set(actual) or calibration.image_id.duplicated().any():
    raise RuntimeError("Calibration identifiers do not match test images")
if not np.isfinite(calibration[["sx", "sy"]]).all().all() or (calibration[["sx", "sy"]] <= 0).any().any():
    raise RuntimeError("Invalid physical scale; inspect audit/scale_calibration.csv")
calibration.to_csv(AUDIT / "scale_calibration.csv", index=False)
print("Synthetic checks passed. Physical scale validated for", len(calibration), "images.")


Synthetic checks passed. Physical scale validated for 309 images.


### 4. Run image inference and produce candidate plus control

In [5]:
START = time.perf_counter()
TAGS = ["test"] + [f"ens_f{k}" for k in range(5)]
PARAMS = {"mt_mode": "centre_minus_half", "fl_mode": "extend_centre", "fl_factor": .9,
          "pa_at": .9, "tex_w": .5, "fl_sin_w": .25}
COLS = ["pa_deg", "fl_mm", "mt_mm"]
for tag in TAGS:
    seg.cached_maps(files, tag)
    print("Cached", tag, "elapsed seconds", round(time.perf_counter() - START, 1), flush=True)

def build_ensemble(label):
    heads = {}
    for tag in TAGS:
        frame = make_sub.build(PARAMS, maps_tag=tag)
        if frame.image_id.duplicated().any() or set(frame.image_id) != set(output_ids):
            raise RuntimeError(f"Invalid model-head identifiers: {tag}")
        frame = frame.set_index("image_id").loc[output_ids].reset_index()
        if not np.isfinite(frame[COLS].to_numpy()).all():
            raise RuntimeError(f"Non-finite model-head output: {tag}")
        frame.to_csv(AUDIT / f"{label}_{tag}_measurements.csv", index=False)
        heads[tag] = frame
    pair = heads["test"][COLS].to_numpy()
    folds = np.stack([heads[tag][COLS].to_numpy() for tag in TAGS[1:]])
    values = .5 * pair + .5 * folds.mean(axis=0)
    result = pd.DataFrame(values, columns=COLS).round(3)
    result.insert(0, "image_id", output_ids)
    if result.image_id.tolist() != output_ids or len(result) != len(files) or not result.image_id.is_unique:
        raise RuntimeError("Submission ID/order mismatch")
    if not np.isfinite(result[COLS].to_numpy()).all():
        raise RuntimeError("Submission contains invalid values")
    for column, (lo, hi) in make_sub.RANGES.items():
        if not result[column].between(lo, hi).all():
            raise RuntimeError(f"Prediction outside inherited clipping range: {column}")
    return result, heads, folds

candidate, candidate_heads, fold_values = build_ensemble("supported")
candidate.to_csv(WORK / "submission.csv", index=False)
# A same-run control isolates this modification; it is not an independently validated score.
bounded_fit = geometry.pa_depth_fit
def unbounded_fit(S, at, min_n=5):
    if len(S) < min_n or np.ptp(S[:, 0]) < .2:
        return None
    return float(np.polyval(np.polyfit(S[:, 0], S[:, 1], 1, w=np.sqrt(S[:, 2])), at))
try:
    geometry.pa_depth_fit = unbounded_fit
    control, _, _ = build_ensemble("unbounded_control")
finally:
    geometry.pa_depth_fit = bounded_fit
control.to_csv(WORK / "control_unbounded.csv", index=False)

delta = candidate[COLS] - control[COLS]
pd.concat([candidate[["image_id"]], delta.add_prefix("delta_")], axis=1).to_csv(AUDIT / "candidate_vs_control.csv", index=False)
spread = pd.DataFrame(fold_values.std(axis=0), columns=["sd_" + c for c in COLS])
spread.insert(0, "image_id", output_ids)
spread.to_csv(AUDIT / "fold_disagreement.csv", index=False)
receipt = {"status": "inference_completed_not_yet_scored", "rows": len(candidate),
    "columns": list(candidate.columns), "candidate_sha256": sha256_file(WORK / "submission.csv"),
    "control_sha256": sha256_file(WORK / "control_unbounded.csv"), "parameters": PARAMS,
    "weights": {"public_pair": .5, "each_of_five_folds": .1},
    "elapsed_seconds": time.perf_counter() - START, "ground_truth_used": False,
    "test_specific_prediction_tables_used": False, "id_order_rule": order_reason,
    "changed_rows": {c: int((delta[c].abs() > 0).sum()) for c in COLS},
    "mean_absolute_change": delta.abs().mean().to_dict(),
    "fallback_rows_by_head": {tag: int((~frame.ok).sum()) for tag, frame in candidate_heads.items()},
    "validation_limit": "No scalar held-out PA/FL/MT labels are supplied; no local accuracy or LB improvement is claimed."}
(WORK / "run_receipt.json").write_text(json.dumps(receipt, indent=2))
print(json.dumps(receipt, indent=2))
print(candidate.head().to_string(index=False))
print("Submission:", WORK / "submission.csv")
import zipfile
with zipfile.ZipFile(WORK / "UMUD_supported_depth_review.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in [WORK / "submission.csv", WORK / "control_unbounded.csv", WORK / "run_receipt.json", WORK / "cache_identity.json"]:
        archive.write(path, path.relative_to(WORK))
    for folder in [AUDIT, WORK / "umud"]:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix in {".json", ".csv", ".py"}:
                archive.write(path, path.relative_to(WORK))
print("Review package:", WORK / "UMUD_supported_depth_review.zip")


test: 50/309
test: 100/309
test: 150/309
test: 200/309
test: 250/309
test: 300/309
Cached test elapsed seconds 47.5
ens_f0: 50/309
ens_f0: 100/309
ens_f0: 150/309
ens_f0: 200/309
ens_f0: 250/309
ens_f0: 300/309
Cached ens_f0 elapsed seconds 93.6
ens_f1: 50/309
ens_f1: 100/309
ens_f1: 150/309
ens_f1: 200/309
ens_f1: 250/309
ens_f1: 300/309
Cached ens_f1 elapsed seconds 139.6
ens_f2: 50/309
ens_f2: 100/309
ens_f2: 150/309
ens_f2: 200/309
ens_f2: 250/309
ens_f2: 300/309
Cached ens_f2 elapsed seconds 185.0
ens_f3: 50/309
ens_f3: 100/309
ens_f3: 150/309
ens_f3: 200/309
ens_f3: 250/309
ens_f3: 300/309
Cached ens_f3 elapsed seconds 231.3
ens_f4: 50/309
ens_f4: 100/309
ens_f4: 150/309
ens_f4: 200/309
ens_f4: 250/309
ens_f4: 300/309
Cached ens_f4 elapsed seconds 279.2
{
  "status": "inference_completed_not_yet_scored",
  "rows": 309,
  "columns": [
    "image_id",
    "pa_deg",
    "fl_mm",
    "mt_mm"
  ],
  "candidate_sha256": "f351ba181d96b710ef1fe6203653343f1556502408bf714df81c7387329ae82e"

### Interpretation after execution

The receipt documents successful image inference and structural checks. Candidate-control differences and fold disagreement are diagnostics, not accuracy estimates. Review the calibration and fallback rows before selecting a competition submission. The public score remains unknown until Kaggle evaluates the candidate; private performance and external generalization remain unknown.


In [7]:
# Post-run metadata audit; does not change predictions or inference modules.
import importlib.metadata as md
import numpy as np
import json, zipfile
numpy_distributions = [{"name": d.metadata["Name"], "version": d.version, "metadata_path": str(d._path)}
    for d in md.distributions() if d.metadata.get("Name", "").lower() == "numpy"]
resolution = {"imported_numpy_version": np.__version__, "imported_numpy_path": np.__file__,
    "artifact_hashes": {name: sha256_file(WORK / name) for name in ["submission.csv", "control_unbounded.csv", "cache_identity.json"]},
    "metadata_numpy_version": md.version("numpy"),
    "numpy_distributions": numpy_distributions,
    "recorded_environment_numpy": json.loads((AUDIT / "environment.json").read_text())["versions"].get("numpy"),
    "cache_identity_numpy": json.loads((WORK / "cache_identity.json").read_text())["dependencies"]["numpy"],
    "explanation": "Environment dictionary kept the last distribution record; runtime import and selected metadata are checked separately."}
assert resolution["imported_numpy_version"] == resolution["metadata_numpy_version"] == resolution["cache_identity_numpy"]
(AUDIT / "dependency_resolution.json").write_text(json.dumps(resolution, indent=2))
with zipfile.ZipFile(WORK / "UMUD_supported_depth_review.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in [WORK / "submission.csv", WORK / "control_unbounded.csv", WORK / "run_receipt.json", WORK / "cache_identity.json"]:
        archive.write(path, path.relative_to(WORK))
    for folder in [AUDIT, WORK / "umud"]:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix in {".json", ".csv", ".py"}:
                archive.write(path, path.relative_to(WORK))
print(json.dumps(resolution, indent=2))
print("Post-run dependency audit completed; predictions unchanged.")


{
  "imported_numpy_version": "2.1.3",
  "imported_numpy_path": "/usr/local/lib/python3.13/dist-packages/numpy/__init__.py",
  "artifact_hashes": {
    "submission.csv": "f351ba181d96b710ef1fe6203653343f1556502408bf714df81c7387329ae82e",
    "control_unbounded.csv": "fa673648414af45c7ab3bbc1c5c1f2db7be90bec34ec7cd65991256a02d95c58",
    "cache_identity.json": "bd8bdf9b334e29c11bc5aaeb29ca9befbc27ffdf8cc89f596a6ef0c0e8e4b17a"
  },
  "metadata_numpy_version": "2.1.3",
  "numpy_distributions": [
    {
      "name": "numpy",
      "version": "2.1.3",
      "metadata_path": "/usr/local/lib/python3.13/dist-packages/numpy-2.1.3.dist-info"
    },
    {
      "name": "numpy",
      "version": "1.26.4",
      "metadata_path": "/usr/lib/python3/dist-packages/numpy-1.26.4.dist-info"
    }
  ],
  "recorded_environment_numpy": "1.26.4",
  "cache_identity_numpy": "2.1.3",
  "explanation": "Environment dictionary kept the last distribution record; runtime import and selected metadata are checked separ

In [8]:
"""Discard invalid model heads and renormalize the existing UMUD ensemble.

This reads model-measurement tables, not labels or another competitor's predictions.
No new inference, training, parameter fitting or leaderboard feedback is used.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

TAGS = ("test", "ens_f0", "ens_f1", "ens_f2", "ens_f3", "ens_f4")
TARGETS = ("pa_deg", "fl_mm", "mt_mm")
BASE_WEIGHTS = np.array([0.5, 0.1, 0.1, 0.1, 0.1, 0.1], dtype=float)
RANGES = {"pa_deg": (5, 45), "fl_mm": (30, 200), "mt_mm": (10, 50)}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def parse_validity(series: pd.Series, head: str) -> np.ndarray:
    """Parse explicit booleans; never interpret the string 'False' as truthy."""
    output = []
    for value in series:
        if isinstance(value, (bool, np.bool_)):
            output.append(bool(value))
        elif isinstance(value, str) and value.strip().lower() in {"true", "false"}:
            output.append(value.strip().lower() == "true")
        else:
            raise ValueError(f"{head}: invalid or missing ok flag: {value!r}")
    return np.asarray(output, dtype=bool)


def combine_heads(head_tables: dict[str, pd.DataFrame], output_ids: list[str]):
    """Return candidate, original ensemble, per-head weights and invalid-head counts.

    A single validity mask and weight vector is used for all three targets.
    All-valid rows use the original arithmetic expression to preserve rounding.
    """
    if set(head_tables) != set(TAGS):
        raise ValueError("Exactly the original pair and all five fold heads are required")
    if not output_ids or len(set(output_ids)) != len(output_ids):
        raise ValueError("Output identifiers must be nonempty and unique")
    values, flags = [], []
    for tag in TAGS:
        frame = head_tables[tag]
        required = {"image_id", "ok", *TARGETS}
        if not required.issubset(frame.columns):
            raise ValueError(f"{tag}: missing required columns")
        if frame.image_id.isna().any() or frame.image_id.duplicated().any():
            raise ValueError(f"{tag}: missing or duplicate image IDs")
        if set(frame.image_id) != set(output_ids):
            raise ValueError(f"{tag}: image ID mismatch")
        frame = frame.set_index("image_id").loc[output_ids]
        numeric = frame[list(TARGETS)].to_numpy(dtype=float)
        if not np.isfinite(numeric).all():
            raise ValueError(f"{tag}: non-finite target value")
        values.append(numeric)
        flags.append(parse_validity(frame.ok, tag))
    values = np.stack(values)  # head, image, target
    valid = np.stack(flags)   # head, image
    active = BASE_WEIGHTS[:, None] * valid
    totals = active.sum(axis=0)
    if (totals <= 0).any():
        failed = [output_ids[i] for i in np.flatnonzero(totals <= 0)]
        raise ValueError(f"No valid model head for: {failed}")
    normalized = active / totals[None, :]
    if not np.allclose(normalized.sum(axis=0), 1., atol=1e-12):
        raise AssertionError("Normalized weights do not sum to one")
    if np.any(normalized[~valid] != 0):
        raise AssertionError("An invalid head retained a positive weight")

    baseline = .5 * values[0] + .5 * values[1:].mean(axis=0)
    combined = baseline.copy()
    affected = (~valid).any(axis=0)
    combined[affected] = (values[:, affected, :] * normalized[:, affected, None]).sum(axis=0)
    if not np.isfinite(combined).all():
        raise ValueError("Non-finite final ensemble")

    def to_frame(array):
        result = pd.DataFrame(array, columns=TARGETS).round(3)
        result.insert(0, "image_id", output_ids)
        return result

    candidate, original = to_frame(combined), to_frame(baseline)
    for column, (low, high) in RANGES.items():
        if not candidate[column].between(low, high).all():
            raise ValueError(f"Final {column} exceeds inherited output range")
    audit = pd.DataFrame([
        {"image_id": image_id, "head": tag, "ok": bool(valid[h, i]),
         "base_weight": float(BASE_WEIGHTS[h]), "active_weight": float(active[h, i]),
         "normalized_weight": float(normalized[h, i]), "invalid_heads_for_image": int((~valid[:, i]).sum())}
        for i, image_id in enumerate(output_ids) for h, tag in enumerate(TAGS)
    ])
    return candidate, original, audit, (~valid).sum(axis=0)


def run(source_dir: str | Path, output_dir: str | Path) -> dict:
    source = Path(source_dir).resolve()
    output = Path(output_dir).resolve()
    if output == source or output == source / "audit":
        raise ValueError("Output must be a separate directory")
    final_names = ("submission.csv", "head_weight_audit.csv", "changed_rows.csv", "receipt.json")
    if any((output / name).exists() for name in final_names):
        raise FileExistsError("Refusing to overwrite existing postprocess outputs")

    parent_path, parent_receipt_path = source / "submission.csv", source / "run_receipt.json"
    parent_hash = sha256_file(parent_path)
    parent_receipt = json.loads(parent_receipt_path.read_text())
    if parent_receipt.get("candidate_sha256") != parent_hash:
        raise ValueError("Parent candidate hash does not match its completed-run receipt")
    if parent_receipt.get("status") != "inference_completed_not_yet_scored":
        raise ValueError("Parent run is not marked as completed image inference")
    parent = pd.read_csv(parent_path)
    if list(parent.columns) != ["image_id", *TARGETS] or parent.image_id.duplicated().any():
        raise ValueError("Parent candidate schema or identifiers are invalid")
    if len(parent) != parent_receipt.get("rows") or not np.isfinite(parent[list(TARGETS)].to_numpy()).all():
        raise ValueError("Parent candidate rows or numeric values do not match its receipt")
    paths = {tag: source / "audit" / f"supported_{tag}_measurements.csv" for tag in TAGS}
    tables = {tag: pd.read_csv(path) for tag, path in paths.items()}
    candidate, reconstructed, audit, failed_counts = combine_heads(tables, parent.image_id.tolist())
    if not np.array_equal(reconstructed[list(TARGETS)].to_numpy(), parent[list(TARGETS)].to_numpy()):
        raise ValueError("Supported-head tables do not exactly reproduce the rounded parent candidate")
    delta = candidate[list(TARGETS)] - parent[list(TARGETS)]
    changed_mask = delta.ne(0).any(axis=1)
    affected_mask = failed_counts > 0
    if changed_mask[~affected_mask].any():
        raise AssertionError("An all-valid row changed")
    changed = candidate.loc[changed_mask, ["image_id"]].copy()
    changed["invalid_head_count"] = failed_counts[changed_mask.to_numpy()]
    for column in TARGETS:
        changed["parent_" + column] = parent.loc[changed_mask, column]
        changed["new_" + column] = candidate.loc[changed_mask, column]
        changed["delta_" + column] = delta.loc[changed_mask, column].round(3)

    output.mkdir(parents=True, exist_ok=True)
    pending = output / "pending"
    pending.mkdir(exist_ok=True)
    candidate.to_csv(pending / "submission.csv", index=False, lineterminator="\n")
    audit.to_csv(pending / "head_weight_audit.csv", index=False, lineterminator="\n")
    changed.to_csv(pending / "changed_rows.csv", index=False, lineterminator="\n")
    input_files = {str(path.relative_to(source)): sha256_file(path) for path in paths.values()}
    input_files["submission.csv"] = parent_hash
    input_files["run_receipt.json"] = sha256_file(parent_receipt_path)
    receipt = {
        "status": "postprocess_completed_not_yet_scored",
        "method": "validity-weighted supported-depth ensemble",
        "rows": len(candidate), "columns": list(candidate.columns),
        "parent_candidate_sha256": parent_hash,
        "candidate_sha256": sha256_file(pending / "submission.csv"),
        "input_sha256": input_files,
        "output_sha256": {name: sha256_file(pending / name) for name in final_names if name != "receipt.json"},
        "validity_rule": "A model head is usable only when its existing ok flag is explicitly True. False receives zero weight for all three targets; remaining base weights are renormalized per image. All-invalid images abort.",
        "base_weights": dict(zip(TAGS, BASE_WEIGHTS.tolist())),
        "same_weights_for_all_targets": True,
        "affected_images": int(affected_mask.sum()), "changed_rows": int(changed_mask.sum()),
        "unchanged_rows": int((~changed_mask).sum()),
        "one_valid_head_images": int((failed_counts == len(TAGS) - 1).sum()),
        "invalid_heads_by_model": {tag: int((~parse_validity(tables[tag].ok, tag)).sum()) for tag in TAGS},
        "parent_exactly_reconstructed": True,
        "all_valid_rows_unchanged": True,
        "output_order_matches_parent": candidate.image_id.tolist() == parent.image_id.tolist(),
        "ground_truth_used": False, "new_model_inference": False,
        "accuracy_claim": "No local accuracy or leaderboard improvement is claimed; no target ground truth was used.",
        "inherited_limits": "Retains parent geometry, calibration assumptions and public-leaderboard-tuned constants; validity correction does not independently validate them. Images supported by only one valid head have no ensemble agreement evidence and must not be treated as confident predictions.",
        "receipt_hash_note": "Receipt is the final completion marker; its own hash is not recursively included.",
    }
    (pending / "receipt.json").write_text(json.dumps(receipt, indent=2) + "\n")
    for name in final_names:  # receipt is promoted last as the commit marker
        if (output / name).exists():
            raise FileExistsError(f"Refusing to replace existing output: {name}")
        (pending / name).rename(output / name)
    pending.rmdir()
    return receipt



SOURCE_WORK = Path("/kaggle/working/umud_supported_depth_v1")
FINAL_WORK = SOURCE_WORK / "failure_aware"
postprocess_receipt = run(SOURCE_WORK, FINAL_WORK)
print(json.dumps(postprocess_receipt, indent=2))
import zipfile
with zipfile.ZipFile(FINAL_WORK / "UMUD_failure_aware_review.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ["submission.csv", "head_weight_audit.csv", "changed_rows.csv", "receipt.json"]:
        archive.write(FINAL_WORK / name, name)
print("Failure-aware candidate:", FINAL_WORK / "submission.csv")
print("Review ZIP:", FINAL_WORK / "UMUD_failure_aware_review.zip")


{
  "status": "postprocess_completed_not_yet_scored",
  "method": "validity-weighted supported-depth ensemble",
  "rows": 309,
  "columns": [
    "image_id",
    "pa_deg",
    "fl_mm",
    "mt_mm"
  ],
  "parent_candidate_sha256": "f351ba181d96b710ef1fe6203653343f1556502408bf714df81c7387329ae82e",
  "candidate_sha256": "953344cd42cb9b358a1572faade92f3c11ef2d710578ff1db9827afd338fb403",
  "input_sha256": {
    "audit/supported_test_measurements.csv": "d26dd0352dc33e47e1b6211eb1947b62d3c689f3fcc2e136e09f640273b882e1",
    "audit/supported_ens_f0_measurements.csv": "a26c4ec02256ec962df60a457450c8fe7374078ab516deac56630d2ebf85aa5a",
    "audit/supported_ens_f1_measurements.csv": "8189dda29df2a2ee24f083c1f3dbeb3fcde3d801fcc4e24da8efceabfdeb30b5",
    "audit/supported_ens_f2_measurements.csv": "c2c1543da28600a5c12745c10774e3f3872a2460a1d275d91c8a08968c6dfec0",
    "audit/supported_ens_f3_measurements.csv": "7ada70439c4e92749309ba7979a602a6594449ee77dc4b2a0d6a574188ec29ed",
    "audit/support